# ASTGCN 云端训练笔记本

本笔记本用于在 Kaggle、Colab 等云端 Ubuntu/Linux 环境中训练当前项目的 ASTGCN recent component。

笔记本中的命令只使用 Python 代码和 Jupyter 的 `!pip`/`!mkdir`/`!cp` 这类 Linux shell magic，不使用 Windows PowerShell 或 conda 命令。

使用前需要准备 PEMS04 数据文件：

- `data/PEMS04/pems04.npz`
- `data/PEMS04/distance.csv`

当前任务边界：使用过去 12 个时间步的第 0 个特征，预测未来 12 个时间步的第 0 个特征。

## 1. 环境与项目根目录

如果在 Kaggle 中运行，建议把本项目完整上传到 `/kaggle/working/ASTGCN`。如果路径不同，只需要修改 `PROJECT_ROOT`。

本单元会打印当前系统类型，正常云端环境通常应显示 `Linux`。

In [ ]:
!git clone https://github.com/Tuzfucius/ASTGCN-learning.git

import os
import sys
import platform
from pathlib import Path

os.chdir('/kaggle/working/ASTGCN-learning')

candidate_roots = [
    Path.cwd(),
    Path('/kaggle/working/ASTGCN'),
    Path('/content/ASTGCN'),
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'src').exists() and (root / 'configurations' / 'PEMS04_astgcn.yaml').exists():
        PROJECT_ROOT = root.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('没有找到项目根目录。请把 PROJECT_ROOT 改成 ASTGCN 项目所在路径。')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('system =', platform.system())
print('python =', sys.version)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('cwd =', Path.cwd())

## 2. 安装与检查依赖

Kaggle 通常已经预装 PyTorch。如果下面的导入失败，再取消注释安装命令。

云端 notebook 中建议使用 `pip`，不要使用本地 Windows 的 conda 环境路径。

In [ ]:
# 如需安装依赖，取消下一行注释。
# !pip install -q numpy PyYAML torch

import numpy as np
import torch
import yaml

print('numpy =', np.__version__)
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device =', torch.cuda.get_device_name(0))

## 3. 读取配置并检查数据

如果使用 Kaggle Dataset，可以先把 `pems04.npz` 和 `distance.csv` 拷贝到 `data/PEMS04/`，也可以直接修改配置中的路径。

Kaggle Dataset 常见路径类似 `/kaggle/input/your-dataset-name/...`。下面给出的是 Ubuntu/Linux shell 示例，按你的真实 Dataset 名称修改后再取消注释。

In [ ]:
# Kaggle Dataset 拷贝示例：按真实路径修改后取消注释。
# !mkdir -p data/PEMS04
# !cp /kaggle/input/your-dataset-name/pems04.npz data/PEMS04/pems04.npz
# !cp /kaggle/input/your-dataset-name/distance.csv data/PEMS04/distance.csv

# 查看当前目录和数据目录，确认路径是否正确。
# !pwd
# !ls -lah
# !ls -lah data/PEMS04

In [ ]:
from src.utils.config import load_config

CONFIG_PATH = 'configurations/PEMS04_astgcn.yaml'
config = load_config(CONFIG_PATH)

# 云端环境中强制按实际 CUDA 可用性设置设备。
config['training']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'

# 如需快速试跑，可临时调小 epoch。例如：
# config['training']['epochs'] = 2

data_dir = Path('data/PEMS04')
data_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    Path(config['data']['graph_signal_matrix_filename']),
    Path(config['data']['adj_filename']),
]

missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError('缺少数据文件：' + ', '.join(missing_files))

print('device =', config['training']['device'])
print('raw data =', config['data']['graph_signal_matrix_filename'])
print('adj file =', config['data']['adj_filename'])
print('processed data =', config['data']['processed_dataset_filename'])

## 4. 数据预处理

该步骤会生成训练、验证和测试数据，保存到配置中的 `processed_dataset_filename`。

In [ ]:
from src.data.preprocessing import generate_dataset, save_dataset

dataset = generate_dataset(config)
save_dataset(dataset, config['data']['processed_dataset_filename'])

for key in ['train_x', 'train_target', 'val_x', 'val_target', 'test_x', 'test_target', 'mean', 'std']:
    print(f'{key}: {dataset[key].shape}')

## 5. 构建模型并训练

最佳验证集权重会保存为 `experiments/PEMS04/astgcn_recent/best.pt`。

In [ ]:
from src.data.dataset import build_all_dataloaders
from src.engine.trainer import Trainer
from src.graph.adjacency import load_adjacency_matrix
from src.graph.laplacian import chebyshev_polynomials, scaled_laplacian
from src.metrics.losses import get_loss_function
from src.models.astgcn import build_astgcn_model
from src.utils.logging import create_experiment_dir, save_config_snapshot
from src.utils.seed import set_seed

set_seed(config['training']['seed'])

output_dir = create_experiment_dir(config)
save_config_snapshot(config, output_dir)

dataloaders = build_all_dataloaders(config)

adj_mx, _ = load_adjacency_matrix(
    config['data']['adj_filename'],
    config['data']['num_of_vertices'],
)
l_tilde = scaled_laplacian(adj_mx)
cheb_polys = chebyshev_polynomials(l_tilde, config['model']['K'])

model = build_astgcn_model(config, cheb_polys)
optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
loss_fn = get_loss_function(config['training']['loss_function'], config['training']['missing_value'])

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    output_dir=output_dir,
    device=config['training']['device'],
)

history = trainer.fit(config['training']['epochs'], config['training']['start_epoch'])
print('output_dir =', output_dir)
print('best_epoch =', None if history['best_epoch'] is None else history['best_epoch'] + 1)
print('best_val_loss =', history['best_val_loss'])

## 6. 测试集评估

该步骤会加载 `best.pt`，计算 MAE、RMSE、MAPE，并按配置保存预测结果。

In [ ]:
from src.engine.evaluator import Evaluator

best_model_path = Path(output_dir) / 'best.pt'
if not best_model_path.exists():
    raise FileNotFoundError(f'没有找到最佳模型权重：{best_model_path}')

evaluator = Evaluator(
    model=model,
    test_loader=dataloaders['test'],
    output_dir=output_dir,
    device=config['training']['device'],
)
evaluator.load_checkpoint(best_model_path)
y_true, y_pred = evaluator.predict()
metrics = evaluator.evaluate(
    y_true,
    y_pred,
    config['training']['metric_method'],
    config['training']['missing_value'],
)

if config['output'].get('save_predictions', True):
    evaluator.save_predictions(y_true, y_pred)

metrics

## 7. 查看和打包结果

Kaggle 会保留 `/kaggle/working` 下的输出。可以把实验目录压缩后下载。

In [ ]:
import shutil

print('实验目录：', output_dir)
for path in sorted(Path(output_dir).iterdir()):
    print(path)

archive_base = Path('astgcn_experiment_result')
archive_path = shutil.make_archive(str(archive_base), 'zip', output_dir)
print('已打包：', archive_path)